# 01 - Integración de los microdatos de la ENDES 2021–2025

Corresponde a la etapa de **Comprensión de los datos** del proceso CRISP-DM (plan de tesis, sección 4.6.1).

Une, por año, tres módulos de la ENDES mediante el identificador del hogar (`HHID`):

| Módulo | Archivo | Variables que aporta |
|---|---|---|
| 1640 – Encuesta de salud | `CSALUD01` | Diagnóstico de diabetes (`QS109`), antropometría, presión arterial, estilo de vida |
| 1630 – Características de la vivienda | `RECH23` | Región natural (`SHREGION`) e índice de riqueza (`HV270`) |
| 1629 – Características del hogar | `RECH0` | Departamento (`HV024`) y área de residencia (`HV025`) |

Aplica los criterios de selección del plan (secciones 4.10 y 4.11) y genera **dos archivos separados**:

- `data/processed/endes_lima_desarrollo_2021_2024.csv`: todo el desarrollo del modelo.
- `data/processed/endes_lima_evaluacion_2025.csv`: **solo** para la evaluación final.

Los microdatos se descargan del portal del INEI (https://proyectos.inei.gob.pe/microdatos/) en la carpeta `ENDES/<año>/<módulo>/`. No se versionan en el repositorio.

In [1]:
import glob
from pathlib import Path

import numpy as np
import pandas as pd

ENDES = Path('../ENDES')
SALIDA = Path('../data/processed')

ANIOS = range(2021, 2026)
ANIO_EVALUACION = 2025
LIMA_METROPOLITANA = 1
EDAD_MIN, EDAD_MAX = 30, 69
CODIGO_NO_VALIDO = 999  # las mediciones antropométricas no realizadas se registran como 999.9

COLS_SALUD = [
    'HHID', 'QSNUMERO', 'QSSEXO', 'QS23', 'QS25N', 'QS26', 'QS102', 'QS107', 'QS109',
    'QS200', 'QS206', 'QS208', 'QS213U', 'QS213C', 'QS219U', 'QS219C',
    'QS900', 'QS901', 'QS903S', 'QS903D', 'QS905S', 'QS905D', 'QS907', 'PESO15_AMAS',
]
COLS_VIVIENDA = ['HHID', 'SHREGION', 'HV270']
COLS_HOGAR = ['HHID', 'HV024', 'HV025']

## Lectura de los archivos

El formato de los CSV publicados por el INEI no es uniforme entre años: el de 2022 usa `;` como separador y coma decimal, y algunos años incluyen marca BOM. La función de lectura detecta ambos casos.

In [2]:
def leer_modulo(patron, columnas):
    rutas = glob.glob(str(patron))
    if len(rutas) != 1:
        raise FileNotFoundError(f'Se esperaba un archivo para {patron}, se encontraron {len(rutas)}')
    with open(rutas[0], 'rb') as f:
        cabecera = f.readline()
    sep = ';' if cabecera.count(b';') > cabecera.count(b',') else ','
    enc = 'utf-8-sig' if cabecera.startswith(b'\xef\xbb\xbf') else 'latin1'
    df = pd.read_csv(rutas[0], sep=sep, encoding=enc, dtype=str, low_memory=False)
    df.columns = [c.strip().upper() for c in df.columns]
    faltantes = set(columnas) - set(df.columns)
    if faltantes:
        raise KeyError(f'{rutas[0]}: faltan columnas {sorted(faltantes)}')
    return df[columnas].apply(lambda s: s.str.strip())


def a_numerico(s):
    return pd.to_numeric(s.str.replace(',', '.', regex=False).replace('', np.nan), errors='coerce')

## Integración por año

In [3]:
def integrar_anio(anio):
    salud = leer_modulo(ENDES / str(anio) / '1640' / 'CSALUD01*.csv', COLS_SALUD)
    vivienda = leer_modulo(ENDES / str(anio) / '1630' / 'RECH23*.csv', COLS_VIVIENDA)
    hogar = leer_modulo(ENDES / str(anio) / '1629' / 'RECH0*.csv', COLS_HOGAR)

    df = salud.merge(vivienda, on='HHID', how='left').merge(hogar, on='HHID', how='left')
    if df['SHREGION'].isna().any():
        raise ValueError(f"{anio}: {df['SHREGION'].isna().sum()} registros sin región tras el cruce")
    for col in df.columns.drop('HHID'):
        df[col] = a_numerico(df[col])
    df.insert(0, 'ANIO', anio)
    return df


datos = pd.concat([integrar_anio(a) for a in ANIOS], ignore_index=True)
datos.groupby('ANIO').size().rename('personas encuestadas (Cuestionario de Salud)').to_frame()

,personas encuestadas (Cuestionario de Salud)
ANIO,
2021,34115
2022,34301
2023,34536
2024,34018
2025,33569


El cruce con los módulos del hogar no pierde ningún registro (la función se detiene si algún registro queda sin región).

## Criterios de selección

- **Población** (sección 4.10): Lima Metropolitana (`SHREGION = 1`), 30 a 69 años y respuesta válida (sí o no) a `QS109`. Se excluye «no sabe» (8).
- **Muestra** (sección 4.11): además, peso, talla y perímetro abdominal medidos con valores válidos (< 999).

In [4]:
lima = datos[(datos['SHREGION'] == LIMA_METROPOLITANA) & datos['QS23'].between(EDAD_MIN, EDAD_MAX)]
poblacion = lima[lima['QS109'].isin([1, 2])]

antropometria_valida = (
    (poblacion['QS900'] < CODIGO_NO_VALIDO)
    & (poblacion['QS901'] < CODIGO_NO_VALIDO)
    & (poblacion['QS907'] < CODIGO_NO_VALIDO)
)
muestra = poblacion[antropometria_valida].copy()
muestra['DM_AUTORREPORTADA'] = (muestra['QS109'] == 1).astype(int)
muestra['IMC'] = muestra['QS900'] / (muestra['QS901'] / 100) ** 2

print(f"Población: {len(poblacion)} personas, {(poblacion['QS109'] == 1).sum()} con diagnóstico autorreportado")
print(f"Muestra:   {len(muestra)} registros")

Población: 11285 personas, 722 con diagnóstico autorreportado
Muestra:   10899 registros


## Separación desarrollo / evaluación final

Según el protocolo de análisis (`docs/protocolo_analisis.md`), los datos de 2025 se reservan para la evaluación final. Por eso, aquí **no se muestran los casos de 2025**: solo el número de registros.

In [5]:
desarrollo = muestra[muestra['ANIO'] < ANIO_EVALUACION]
evaluacion = muestra[muestra['ANIO'] == ANIO_EVALUACION]

resumen = desarrollo.groupby('ANIO').agg(registros=('ANIO', 'size'), casos=('DM_AUTORREPORTADA', 'sum'))
resumen.loc['Total'] = resumen.sum()
print(f'Conjunto de evaluación ({ANIO_EVALUACION}): {len(evaluacion)} registros')
resumen

Conjunto de evaluación (2025): 1990 registros


,registros,casos
ANIO,,
2021,2357,142
2022,2256,136
2023,2135,155
2024,2161,129
Total,8909,562


In [6]:
SALIDA.mkdir(parents=True, exist_ok=True)
desarrollo.to_csv(SALIDA / 'endes_lima_desarrollo_2021_2024.csv', index=False)
evaluacion.to_csv(SALIDA / f'endes_lima_evaluacion_{ANIO_EVALUACION}.csv', index=False)
print('Archivos escritos en', SALIDA.resolve())

Archivos escritos en C:\Users\Renzo\Desktop\diabetes-prediction-system\data\processed
